# Fig. S24 | Equal-volume spatial strategies

Plots storage benefits for three pumping-reduction strategies.

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p.resolve() for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'management').is_dir())
DATA = ROOT / 'outputs' / 'MANAGEMENT_2012_2013' / 'equal_volume'
OUT = ROOT / 'outputs' / 'figures' / 'FigS24' / 'FigS24.png'
OUT.parent.mkdir(parents=True, exist_ok=True)
metrics = pd.read_csv(DATA / 'drought_strategy_seed_metrics.csv', dtype={'seed':str})
strategies = ('Uniform','High pumping','Leverage guided')
colors = {'Uniform':'#4A4A4A','High pumping':'#7489B5','Leverage guided':'#16857C'}
labels = {'Uniform':'Uniform','High pumping':'High pumping','Leverage guided':'Leverage-guided'}
PI75 = 1.150349
mpl.rcParams.update({'font.family':'Arial','font.size':10,'axes.labelsize':11,'axes.titlesize':11,'xtick.labelsize':9,'ytick.labelsize':9,'axes.linewidth':0.7,'axes.spines.top':True,'axes.spines.right':True})

In [ ]:
def summarize(column):
    rows = []
    for (strategy, budget), group in metrics.groupby(['strategy','budget_nominal']):
        values = group[column].to_numpy(float)
        mean = values.mean(); radius = PI75 * values.std(ddof=0)
        rows.append((strategy, budget, group.actual_dV_m3.mean(), mean, mean-radius, mean+radius))
    return pd.DataFrame(rows, columns=['strategy','budget','volume','mean','low','high'])

panels = [('benefit_201210_m3','a  Intervention end'),('benefit_201212_m3','b  Two months later'),('benefit_201304_m3','c  Six months later')]
fig, axes = plt.subplots(1,3,figsize=(9.0,2.8),sharey=True)
for ax, (column,title) in zip(axes,panels):
    summary = summarize(column)
    for strategy in strategies:
        d = summary[summary.strategy.eq(strategy)].sort_values('volume')
        x = d.volume.to_numpy()/1e9; y = d['mean'].to_numpy()/1e9
        ax.fill_between(x,d.low.to_numpy()/1e9,d.high.to_numpy()/1e9,color=colors[strategy],alpha=0.14,lw=0)
        ax.plot(x,y,color=colors[strategy],lw=1.8,label=labels[strategy])
    ax.set_title(title,loc='left',fontweight='bold'); ax.set_xlabel(r'Pumping reduction ($10^9$ m$^3$)')
    ax.set_xlim(0,5.15); ax.set_xticks([0,1,2,3,4,5]); ax.grid(axis='y',color='#ECE9E4',lw=0.5)
axes[0].set_ylabel(r'Storage benefit ($10^9$ m$^3$)')
axes[0].legend(frameon=False,fontsize=8,loc='upper left')
fig.tight_layout(w_pad=1.0)
fig.savefig(OUT,dpi=600,bbox_inches='tight',facecolor='white')
plt.show()